In [364]:

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.base import clone
from sklearn.experimental import enable_iterative_imputer   # noqa: F401 (needed before importing IterativeImputer)
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, roc_auc_score

DATA_DIR = Path("data")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE) 

import warnings
warnings.filterwarnings('ignore')

# CSV Import

In [365]:
train_raw = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["purchase_date"]).set_index("ID")
test_raw = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["purchase_date"]).set_index("ID")
clientes = pd.read_csv(DATA_DIR / "clientes.csv")

target = 'repeat_purchase_90d'

# Sorting by Date

Since we are doing a TimesSeriesSplit we need to sort by `purchase_date`

In [366]:
train_raw = train_raw.sort_values(['purchase_date', 'ID'])  # ID breaks ties on the same day
test_raw = test_raw.sort_values(['purchase_date', 'ID'])

# Quick Eda and Data Types

In [367]:
train_raw.head(3)

,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,...,spend_365d,customer_id,payment_type,salesperson,commercial_zone,transport_method,warehouse,document_series,purchase_date,random_noise
ID,,,,,,,,,,,,,,,,,,,,,
10000812,55.86,51.99,51.99,3.87,4.39,0.00,0.0,28.00,1.0,1.0,...,0.0,120503,Maybe Tomorrow Payment 12,VLOOKUP Master 1,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 49,Final Excel Warehouse 1,Ctrl Z Series 3,2015-07-11,-0.012756
10007082,268.76,436.99,218.50,50.26,0.00,218.49,0.0,116.14,9.0,1.0,...,0.0,113110,Maybe Tomorrow Payment 12,NaN,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 80,Final Excel Warehouse 1,Ctrl Z Series 3,2015-07-11,-0.503701
10000232,541.92,508.20,508.20,33.72,0.00,0.00,0.0,285.70,6.0,1.0,...,0.0,568558,Maybe Tomorrow Payment 7,VLOOKUP Master 1,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 10,Final Excel Warehouse 1,Ctrl Z Series 3,2015-07-12,1.496488


In [368]:
print(train_raw.shape)
train_raw.info()
train_raw.isna().mean()

(6534, 38)
<class 'pandas.DataFrame'>
Index: 6534 entries, 10000812 to 10007590
Data columns (total 38 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   Total a Pagar                   6469 non-null   float64       
 1   Total Bruto                     6469 non-null   float64       
 2   Total Liquido                   6469 non-null   float64       
 3   Total Imp_                      6469 non-null   float64       
 4   Total Portes                    6469 non-null   float64       
 5   Total Desc_ Global              6469 non-null   float64       
 6   Total Desc_ Linha               6469 non-null   float64       
 7   Custo                           6469 non-null   float64       
 8   N_ Detalhes                     6469 non-null   float64       
 9   invoice_count_current_purchase  6469 non-null   float64       
 10  repeat_purchase_90d             6534 non-null   int64         
 11

Total a Pagar                     0.009948
Total Bruto                       0.009948
Total Liquido                     0.009948
Total Imp_                        0.009948
Total Portes                      0.009948
Total Desc_ Global                0.009948
Total Desc_ Linha                 0.009948
Custo                             0.009948
N_ Detalhes                       0.009948
invoice_count_current_purchase    0.009948
repeat_purchase_90d               0.000000
days_since_previous_purchase      0.131772
customer_tenure_days              0.009948
prior_purchase_count              0.009948
prior_total_spend                 0.009948
prior_mean_purchase_value         0.131925
prior_std_purchase_value          0.170340
prior_min_purchase_value          0.102234
prior_max_purchase_value          0.102234
prior_mean_gap_days               0.200184
prior_std_gap_days                0.226201
purchase_count_30d                0.009948
spend_30d                         0.009948
purchase_co

### Removing empty records

- We decided to simply remove the 65 rows since they only had customer_id and repeat_purchase_90d and no other record.

In [369]:
empty = train_raw.drop(columns=['customer_id', target]).isna().all(axis=1)
train = train_raw.loc[~empty].copy()

print(f'empty rows removed from train: {empty.sum()}')
print(f'duplicated rows left in train: {train.duplicated().sum()}')
print(f'train now: {train.shape}')

empty rows removed from train: 65
duplicated rows left in train: 0
train now: (6469, 38)


- the duplicated rows where also part of these 65 empty rows

In [370]:
train.duplicated().sum()

np.int64(0)

In [371]:
train['purchase_date'].is_monotonic_increasing

True

### Impossible and Missing Values

In [372]:
sentinel = 1e9
col_impossible = ['days_since_previous_purchase', 'prior_mean_gap_days', # columns with impossible values found in eda
                'prior_mean_purchase_value', 'spend_365d']

def clean_data(df):
    df=df.copy()
# removes all negative and sentinel values
    for col in col_impossible:
        incorrect = (df[col]<0) | (df[col] >= sentinel)
        df.loc[incorrect, col] = np.nan

# corrects values that do not make sense, like prior_max_purchase_value should never be higher than prior_total_spend
    incorrect = df['prior_max_purchase_value'] > df['prior_total_spend'] +0.01
    df.loc[incorrect,'prior_max_purchase_value']=np.nan

# we need to set the missing prior_mean_purchase_value using prior_count and prior_total where we can
    count = df['prior_purchase_count']
    has_prior = count > 0 # only when prior_purchase_count exists can we use it
    new_mean = df['prior_total_spend']/count.where(has_prior)
    df['prior_mean_purchase_value']=new_mean

# if the customer only has 1 purchase (prior_purchase_count=1) then we can fill day_since_prior_purchase with customer_tenure_days
    is_to_fill = (count ==1) & df['days_since_previous_purchase'].isna()
    df.loc[is_to_fill, 'days_since_previous_purchase'] = df.loc[is_to_fill, 'customer_tenure_days']
    return df


def check_miss_imp(df_train, df_test):
    numeric = [c for c in df_train.select_dtypes(include="number").columns if c != target]
    found = False
    for column in [c for c in numeric if c not in ("customer_id", "random_noise")]:
        flags = {split: {"== 1e9": int((frame[column] >= 1e9).sum()),
                        "< 0": int((frame[column] < 0).sum())}
                for split, frame in [("train", df_train), ("test", df_test)]}
        if any(flags["train"].values()) or any(flags["test"].values()):
            print(f"{column:30s} train {flags['train']}  test {flags['test']}")
            found = True
    for split, frame in [("train", df_train), ("test", df_test)]:
        n = int((frame['prior_max_purchase_value'] > frame['prior_total_spend'] + 0.01).sum())
        if n:
            print(f"prior_max > prior_total_spend   {split}: {n}")
            found = True
    if not found:
        print("no impossible values left in train or test")

In [373]:
check_miss_imp(train, test_raw)


days_since_previous_purchase   train {'== 1e9': 0, '< 0': 16}  test {'== 1e9': 0, '< 0': 2}
prior_mean_purchase_value      train {'== 1e9': 16, '< 0': 16}  test {'== 1e9': 2, '< 0': 2}
prior_mean_gap_days            train {'== 1e9': 0, '< 0': 16}  test {'== 1e9': 0, '< 0': 2}
spend_365d                     train {'== 1e9': 15, '< 0': 15}  test {'== 1e9': 2, '< 0': 2}
prior_max > prior_total_spend   train: 45
prior_max > prior_total_spend   test: 6


In [374]:
train = clean_data(train)
test = clean_data(test_raw)
check_miss_imp(train, test)

no impossible values left in train or test


# Feature Engineering

From EDA we know that repeat purchase is more due to recency and frequency, so we are going to create features that show these factors. 
    - we wan to check if having at least 1 purchase has any correlation with repeat pruchases.
    - for recency we will check for purchases within the least amount of time (30 and 90 days)
    - for frquency we want to check for purchases troughout the year 
    - we also need to check if the share of purchases being recent shows any correlation
    - and finally we want to check if having a higher discount helps with repeat purchase

In [375]:
new_features = ['has_purchased_before', 'bought_30d', 'bought_90d', 'purchases_per_year', 'recent_share_30d', 'recent_share_90d', 'discount_rate']

def add_features(df):
    df = df.copy()
    count = df['prior_purchase_count']

    df['has_purchased_before'] = (count > 0).astype(int)     
    df['bought_30d'] = (df['purchase_count_30d'] > 0).astype(int)
    df['bought_90d'] = (df['purchase_count_90d'] > 0).astype(int)
    df['purchases_per_year'] = count / (df['customer_tenure_days'] + 1) * 365
    df['recent_share_30d'] = df['purchase_count_30d'] / df['purchase_count_365d']
    df['recent_share_90d'] = df['purchase_count_90d'] / df['purchase_count_365d']
    df['discount_rate'] = df['Total Desc_ Linha'] / df['Total Bruto']

# a division by zero gives inf; turn it into missing so the imputer handles it later
    df[new_features] = df[new_features].replace([np.inf, -np.inf], np.nan)
    return df

train = add_features(train)
test = add_features(test)

check = train[new_features + ['random_noise']].corrwith(train[target], method='spearman')
print(check.sort_values(key=abs, ascending=False).round(3).to_string())
print('\nmissing values in the new features (train and test):')
print(pd.DataFrame({'train': train[new_features].isna().sum(), 'test': test[new_features].isna().sum()}).to_string())


purchases_per_year      0.420
bought_90d              0.327
bought_30d              0.234
discount_rate           0.222
recent_share_90d        0.177
has_purchased_before    0.146
recent_share_30d        0.141
random_noise           -0.008

missing values in the new features (train and test):
                      train  test
has_purchased_before      0     0
bought_30d                0     0
bought_90d                0     0
purchases_per_year        0     0
recent_share_30d        843    59
recent_share_90d        843    59
discount_rate             0     0


- checking for correlation between the new features and the target agains random_noise that `purchases_per_year`and both recency features show the highest correlation

# Cliente.csv

In [376]:
#  Only the columns worth testing (EDA 1.9); the rest are empty or act as identifiers
client_cols = ["country_or_region", "commercial_region", "customer_since_year"]
clients_small = clientes_raw.set_index("customer_id")[client_cols]

train = train.join(clients_small, on="customer_id")
test = test.join(clients_small, on="customer_id")

for df in (train, test):
    df["customer_age_years"] = df["purchase_date"].dt.year - df["customer_since_year"]

# Identifiers drop and Data split

In [377]:
# 1. Keep the dates aside: TimeSeriesSplit needs them, the model does not
dates_train = train['purchase_date']
print('train sorted by date:', dates_train.is_monotonic_increasing)

# 2. Columns that are not used as features (reasons in the markdown above)
cols_to_drop = ['customer_id',          # label, not a measure (EDA 1.7)
                'purchase_date',        # test dates lie beyond train (EDA 1.8)
                'commercial_zone',      # 96.8% one level, 209 test rows unseen (EDA 1.8)
                'warehouse',            # 99.7% one level (EDA 1.3)
                'document_series',      # same repeat rate in every level (EDA 1.11)
                'customer_since_year']  # replaced by customer_age_years

# 3. Split features and target; test gets exactly the same columns, in the same order
y = train[target]
X_train = train.drop(columns=[target] + cols_to_drop)
X_test = test[X_train.columns]

print('X:', X_train.shape, ' X_test:', X_test.shape)
print('same columns in train and test:', list(X_train.columns) == list(X_test.columns))

# TimeSeriesSplit with 90 day hor
def time_splits_with_horizon(dates, n_splits=5, horizon_days=90):
    for tr_idx, va_idx in TimeSeriesSplit(n_splits=n_splits).split(dates):
        va_start = dates.iloc[va_idx[0]]
        keep = dates.iloc[tr_idx] < va_start - pd.Timedelta(days=horizon_days)
        yield tr_idx[keep.to_numpy()], va_idx

cross_v = list(time_splits_with_horizon(dates_train))

# Check: each fold trains on the past and validates on the future
folds = pd.DataFrame([{'train rows': len(tr), 'train ends': dates_train.iloc[tr].max().date(),
                       'valid starts': dates_train.iloc[va].min().date(), 'valid ends': dates_train.iloc[va].max().date(),
                       'valid rows': len(va), 'valid repeat rate': round(y.iloc[va].mean(), 3)}
                      for tr, va in cross_v])
print(folds.to_string())

train sorted by date: True
X: (6469, 42)  X_test: (986, 42)
same columns in train and test: True
   train rows  train ends valid starts  valid ends  valid rows  valid repeat rate
0         940  2017-05-12   2017-08-12  2019-01-12        1078              0.584
1        1922  2018-10-13   2019-01-12  2020-10-03        1078              0.534
2        3109  2020-07-04   2020-10-03  2022-03-05        1078              0.576
3        4189  2021-12-04   2022-03-05  2023-09-25        1078              0.520
4        5207  2023-06-24   2023-09-25  2025-02-15        1078              0.604


In [378]:
X_train.head(3)

,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,...,has_purchased_before,bought_30d,bought_90d,purchases_per_year,recent_share_30d,recent_share_90d,discount_rate,country_or_region,commercial_region,customer_age_years
ID,,,,,,,,,,,,,,,,,,,,,
10000812,55.86,51.99,51.99,3.87,4.39,0.00,0.0,28.00,1.0,1.0,...,0,0,0,0.0,NaN,NaN,0.0,Country CT10: Land of The Final Version,Region RG17: We Will See Later,0
10007082,268.76,436.99,218.50,50.26,0.00,218.49,0.0,116.14,9.0,1.0,...,0,0,0,0.0,NaN,NaN,0.0,Country CT10: Land of The Final Version,NaN,0
10000232,541.92,508.20,508.20,33.72,0.00,0.00,0.0,285.70,6.0,1.0,...,0,0,0,0.0,NaN,NaN,0.0,Country CT10: Land of The Final Version,Region RG17: We Will See Later,0


# Preprocessing Pipeline

In [379]:
# 1. Column groups, by name
flag_cols = ['has_purchased_before', 'bought_30d', 'bought_90d']             # already 0/1, never missing
low_cat_cols = ['payment_type', 'salesperson', 'country_or_region', 'commercial_region']
high_cat_cols = ['transport_method']                                              # 80 levels
plain_num_cols = ['customer_tenure_days', 'customer_age_years', 'discount_rate',
                   'recent_share_30d', 'recent_share_90d', 'random_noise']              # skew below 2
skewed_num_cols = [c for c in X_train.columns                                           # every other numeric column
                   if c not in flag_cols + low_cat_cols + high_cat_cols + plain_num_cols]

# 2. One small pipeline per group
skewed_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),                                 # median: robust to long tails
    ('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),       # removes the right skew
    ('scale', StandardScaler()),
])
plain_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])
def category_pipe(min_rows):
    return Pipeline([
        ('impute', SimpleImputer(strategy='constant', fill_value='missing')),     # "missing" becomes a level
        ('encode', OneHotEncoder(handle_unknown='infrequent_if_exist',            # unseen test levels -> rare group
                                 min_frequency=min_rows, sparse_output=False)),   # levels below min_rows -> rare group
    ])

# 3. Put the groups together; nothing is fitted yet
preprocessor = ColumnTransformer([
    ('skewed', skewed_pipe, skewed_num_cols),
    ('plain', plain_pipe, plain_num_cols),
    ('flags', 'passthrough', flag_cols),
    ('low_cat', category_pipe(20), low_cat_cols),
    ('transport', category_pipe(50), high_cat_cols),
], remainder='drop').set_output(transform='pandas')

# Check only: fit on train, apply to test (the real fitting happens inside each CV fold)
X_check = preprocessor.fit_transform(X_train)
X_test_check = preprocessor.transform(X_test)
print('every column assigned to a group:', set(X_train.columns) == set(skewed_num_cols + plain_num_cols + flag_cols
                                                                  + low_cat_cols + high_cat_cols))
print('train:', X_check.shape, ' test:', X_test_check.shape)
print('missing values left (train / test):', int(X_check.isna().sum().sum()), '/', int(X_test_check.isna().sum().sum()))
print('same output columns:', list(X_check.columns) == list(X_test_check.columns))

every column assigned to a group: True
train: (6469, 100)  test: (986, 100)
missing values left (train / test): 0 / 0
same output columns: True


# Handoff

In [380]:
import joblib

# Everything the feature-selection and modelling notebooks need, in one file
handoff = {
    'X_train': X_train,              # 6,469 rows, cleaned, with new features
    'y': y,                          # target, same row order as X_train
    'X_test': X_test,                # 986 rows, same columns, in sample_submission order
    'dates_train': dates_train,      # purchase dates of X_train, to rebuild time splits if needed
    'cross_v': cross_v,              # the 5 time-ordered folds with the 90-day gap
    'preprocessor': clone(preprocessor),   # the recipe, UNFITTED: each fold refits it
    'column_groups': {'skewed': skewed_num_cols, 'plain': plain_num_cols, 'flags': flag_cols,
                      'low_cat': low_cat_cols, 'high_cat': high_cat_cols},
}

output_file = DATA_DIR / 'preprocessed.joblib'
joblib.dump(handoff, output_file)
print('saved to', output_file)
print({name: getattr(obj, 'shape', len(obj) if hasattr(obj, '__len__') else type(obj).__name__)
       for name, obj in handoff.items()})

saved to data/preprocessed.joblib
{'X_train': (6469, 42), 'y': (6469,), 'X_test': (986, 42), 'dates_train': (6469,), 'cross_v': 5, 'preprocessor': 'ColumnTransformer', 'column_groups': 5}
